In [1]:
import numpy as np
import sys, os
import matplotlib.pyplot as plt
from scipy.constants import lambda2nu, nu2lambda, c
lumapiFile = "C:\\Program Files\\Lumerical\\v221\\api\\python\\lumapi.py"

import importlib.util
import sys

spec = importlib.util.spec_from_file_location("lumapi", lumapiFile)
lumapi = importlib.util.module_from_spec(spec)
sys.modules["lumapi"] = lumapi
spec.loader.exec_module(lumapi)

import lumapi

C:\Program Files\Lumerical\v221\api\python\lumapi.py:796: SyntaxWarning: invalid escape sequence '\s'
  message = re.sub('^(Error:)\s(prompt line)\s[0-9]+:', '', str(rvals[2])).strip()


In [2]:
from IPython.core.display import HTML
from IPython.display import display, Math
from IPython.core.pylabtools import figsize

HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

figsize(8, 4)

In [3]:
modeApi = lumapi.MODE(hide = False)

#units

um = 1e-6
nm = 1e-9

## Especificando as dimensões do dispositivo e materiais ##

In [4]:
# Materials:
material = 'Si (Silicon) - Palik'
sub_material = 'SiO2 (Glass) - Palik'

# Dimensões:

wvg_width = 450 * nm
wvg_height = 220 * nm
WM = 7 * um
L_taper = 15 * um
taper_max_height = 1.5 * um

### Calculo do comprimento da caixa central ###

In [5]:
wavelength = 1550 * nm
n_c = 1.5
n_r = 3.5 # Initial value
sigma = 0
N = 2

We = WM + (wavelength / np.pi) * np.power(n_c / n_r,  2*sigma) * np.power(n_r**2 - n_c**2, -0.5)
L_pi = 4*n_r*We**2 / (3*wavelength)
L = 3*L_pi/(4*N)
print('\33[1m Comprimento da caixa central: L = %.2f \u03BCm.\033[0m' %(L/um))

 Comprimento da caixa central: L = 57.82 μm.


## Construção do dispositivo ##

In [ ]:
modeApi.addrect()
modeApi.set('name', 'central box')
modeApi.set('x', 0)
modeApi.set('x span', L)
modeApi.set('y', 0)
modeApi.set('y span', WM)
modeApi.set('z', 0)
modeApi.set('z span', wvg_height)
modeApi.set('material', material)

x_tapers = [(L+L_taper)/2, (L+L_taper)/2, -(L+L_taper)/2, -(L+L_taper)/2]
y_tapers = [We/6, -We/6, We/6, -We/6]
width_l_tapers = [taper_max_height, taper_max_height, wvg_width, wvg_width]
width_r_tapers = [wvg_width, wvg_width, taper_max_height, taper_max_height]

for i in range(0,4,1):
    modeApi.addobject('linear_taper')
    modeApi.set('name', 'taper_%d' %(i+1))
    modeApi.set('x', x_tapers[i])
    modeApi.set('len', L_taper)
    modeApi.set('y', y_tapers[i])
    modeApi.set('width_l', width_l_tapers[i])
    modeApi.set('width_r', width_r_tapers[i])
    modeApi.set('z', 0)
    modeApi.set('thickness', wvg_height)
    modeApi.set('angle_side', 90)
    modeApi.set('material', material)

In [18]:
wvg_length = 3*um

x_guias = [(L+2*L_taper+wvg_length)/2, (L+2*L_taper+wvg_length)/2, -(L+2*L_taper+wvg_length)/2, -(L+2*L_taper+wvg_length)/2]

for i in range(0,4,1):
    modeApi.addrect()
    modeApi.set('name', 'guia_%d' %(i+1))
    modeApi.set('x', x_guias[i])
    modeApi.set('x span', wvg_length)
    modeApi.set('y', y_tapers[i])
    modeApi.set('y span', wvg_width)
    modeApi.set('z', 0)
    modeApi.set('z span', wvg_height)
    modeApi.set('material', material)

## Definindo o solver FDE ##

In [26]:
x_fde = 0
y_fde = 0
z_fde = 0
x_span_fde = 0
y_span_fde = 5*WM
z_span_fde = 5*wvg_height
mesh_cells = 100
wavelength = 1550*nm
start_wavelength = 1500*nm
stop_wavelength = 1600*nm
mesh_multiplier = 5

In [21]:
# Add solver
modeApi.addfde()
modeApi.set('solver type', '2D X normal')
modeApi.set('background material', sub_material)
modeApi.set('x', x_fde)
#modeApi.set('x span', x_span_fde)
modeApi.set('y', y_fde)
modeApi.set('y span', y_span_fde)
modeApi.set('z', z_fde)
modeApi.set('z span', z_span_fde)
modeApi.set('define y mesh by', 'number of mesh cells')
modeApi.set('define z mesh by', 'number of mesh cells')
modeApi.set('mesh cells y', mesh_cells)
modeApi.set('mesh cells z', mesh_cells)

modeApi.set('wavelength', wavelength)

modeApi.set('fit materials with multi-coefficient model', True)
modeApi.set('wavelength start', start_wavelength)
modeApi.set('wavelength stop', stop_wavelength)

modeApi.set('y min bc', 'PML')
modeApi.set('y max bc', 'PML')
modeApi.set('z min bc', 'PML')
modeApi.set('z max bc', 'PML')

#add mesh
modeApi.addmesh()
modeApi.set('set mesh multiplier', True)
modeApi.set('x', x_fde)
modeApi.set('x span', 0)
modeApi.set('x mesh multiplier', 1)
modeApi.set('y', y_fde)
modeApi.set('y span', WM)
modeApi.set('y mesh multiplier', mesh_multiplier)
modeApi.set('z', z_fde)
modeApi.set('z span', wvg_height)
modeApi.set('z mesh multiplier', mesh_multiplier)

In [22]:
number_of_modes = int(modeApi.findmodes())
print(number_of_modes)

28


In [ ]:
n_eff_matrix = []

#coletando os indices efetivos dos modos propagados:

for i in range(1, number_of_modes+1, 1):
    n_eff_matrix.append((modeApi.getresult('FDE::data::mode%d' %i, 'neff').real)[0,0])

In [24]:
print(n_eff_matrix)

[np.float64(2.8457265370400475), np.float64(2.8393056420143643), np.float64(2.828575028181768), np.float64(2.813490346575565), np.float64(2.793988143863045), np.float64(2.7699844172672745), np.float64(2.741372634376433), np.float64(2.708021112066614), np.float64(2.6697696076055966), np.float64(2.6264249202671826), np.float64(2.5777552280643627), np.float64(2.5234827848096693), np.float64(2.4632744709019936), np.float64(2.3967295256091616), np.float64(2.3233636117125367), np.float64(2.2425882835204174), np.float64(2.1536853499304165), np.float64(2.0670963997192198), np.float64(2.058689016083638), np.float64(2.0557779976407047), np.float64(2.044618620738612), np.float64(2.024796816630769), np.float64(1.9990973846244409), np.float64(1.967354027483946), np.float64(1.9478103609733064), np.float64(1.828587790918013), np.float64(1.6971208061280894), np.float64(1.5546992891984868)]


In [ ]:
np.savetxt('neff_core.txt', n_eff_matrix, delimiter=',') # Saving the effective index of the modes in a .txt file